# Pix2Gif replication: motion-guided diffusion for GIF generation

**Paper:** Kandala, Gao, Yang. *Pix2Gif: Motion-Guided Diffusion for GIF Generation*, [arXiv 2403.04634](https://arxiv.org/abs/2403.04634) (Microsoft Research)

**Idea.** Treat GIF generation as **image-to-image translation** (InstructPix2Pix style). Given a source image, a caption and a
**motion magnitude** *m*, the model predicts the frame that is *m* pixels of optical flow away. A GIF is the same source image
rendered at increasing *m* (e.g. 2, 4, 6, 8, 11, 13, 16, 19).

**Architecture** (final "Warp-concat" model):
1. Caption + `" The optical flow is <m in words>."` → CLIP → **c_T**. Motion *m* → learned embedding → **c_M**. **c_L = c_T + c_M**.
2. **FlowNet** (a small UNet; c_L is injected at 16×16) predicts a flow field over the source latent *E(c_I)*. **WarpNet**
   (`grid_sample`) warps it into **z_W**.
3. The SD-1.5 UNet denoises with input **[z_t, E(c_I), z_W]** (12 channels) and cross-attention context c_L.
4. Loss: **L_LDM + 0.01 · L_perceptual(z_W, E(c_I))**, where the perceptual term uses a VGG19 whose first conv is widened to 4 channels so it runs on latents.
5. Data: TGIF frame pairs whose Lucas–Kanade mean flow is in **[2, 20]**, with ≥10 pairs per GIF (783k pairs in the paper).

**Provenance.** The authors deleted the code from [hiteshK03/Pix2Gif](https://github.com/hiteshK03/Pix2Gif) in Sept 2024. This notebook
ports it from commit **`781ca78`**, the last commit that still has it (cloned in cell 1 for reference). **No weights were ever released**, and the TGIF
pair-mining script was never published. So this notebook rebuilds the dataset from the paper's Algorithm 1 and **trains the model**.

| | Paper / official code | This notebook |
|---|---|---|
| FlowNet, WarpNet, InjectionModule, motion embedding, prompt template | ✓ | ported line by line |
| 12-ch UNet input, `hybrid` conditioning (context = c_T + c_M) | ✓ | ✓ |
| Perceptual loss (VGG19-4ch, layers 0/5/10/19/28, λ = 0.01) | ✓ | ✓ |
| CFG dropout 5% text / 5% image / 5% both; AdamW lr 1e-4; 256²; LitEma | ✓ | ✓ |
| Sampler: Euler ancestral, text CFG 7.5, image CFG 1.8, per-frame re-seed | ✓ (`demo.py`) | ✓ |
| Init | SD-1.5 + ft-MSE VAE | **InstructPix2Pix** by default (a big head start on Colab compute); SD-1.5 is selectable |
| Data scale | ~100k GIFs → 783k pairs | `N_GIFS` (default 3k GIFs → ~25k pairs) |
| Compute | 16×V100, eff. batch 256, ~24k steps | 1 GPU, eff. batch 32, `MAX_STEPS` (default 5k), bf16, 8-bit AdamW, gradient checkpointing |
| LK threshold / feature parameters | not stated | `FLOW_EPS = 0.5`, OpenCV defaults (flow measured at 256²) |

**Runtime:** Colab **A100** (recommended) or **L4**. A T4 doesn't have enough memory for full UNet fine-tuning. With the defaults, building the
dataset takes about 10 min and training about 1–2 h on an A100 (an estimate, not measured). Everything persists on Google Drive, so a dropped
session resumes where it stopped. Expect lower quality than the paper at this scale. Raise `N_GIFS` and `MAX_STEPS` to close the gap.

**Run order:** 1 → 2 → 3 → 4 → 5 (train) → 6 (make GIFs) → 7 (evaluate).

In [ ]:
#@title 1 · Setup: GPU check, installs, Drive, official code
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install -U diffusers transformers accelerate bitsandbytes inflect opencv-python-headless safetensors
import os, subprocess, torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU (A100 recommended, L4 works)"

USE_DRIVE = True  #@param {type:"boolean"}
if USE_DRIVE:  # Colab sessions die; keep the dataset tarball + checkpoints on Drive
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/pix2gif"
else:
    ROOT = "/content/pix2gif"
os.makedirs(ROOT, exist_ok=True)

# Official code, for reference. The authors deleted it from the repo in Sept 2024 (commit 0be59b6);
# 781ca78 is the last commit that still has it. The model code below is ported from these files:
#   stable_diffusion/ldm/modules/diffusionmodules/openaimodel.py  (FlowNet, WarpNet, InjectionModule, IntervalEmbedding)
#   stable_diffusion/ldm/modules/diffusionmodules/util.py         (conv / deconv / i_conv / predict_flow)
#   stable_diffusion/ldm/models/diffusion/ddpm_edit.py           (conditioning, CFG dropout, losses)
#   stable_diffusion/ldm/modules/losses/perceptual.py            (4-channel VGG19 perceptual loss)
#   edit_dataset.py, demo.py, demo_parallel.py, configs/*.yaml
if not os.path.exists("/content/Pix2Gif-official"):
    subprocess.run("git clone -q https://github.com/hiteshK03/Pix2Gif.git /content/Pix2Gif-official"
                   " && cd /content/Pix2Gif-official && git checkout -q 781ca7859e", shell=True, check=True)
print("official code at /content/Pix2Gif-official ·", torch.cuda.get_device_name(0))

In [ ]:
#@title 2 · Config
# ---- data (paper: 783,184 train pairs from TGIF) ----
N_GIFS = 3000  #@param {type:"integer"}
PAIRS_PER_GIF = 10         # paper: ">= 10 pairs per GIF (or all, if fewer)"
MOTION_RANGE = (2, 20)     # paper Sec. 3.3: keep pairs whose mean flow magnitude is in [2, 20]
FLOW_EPS = 0.5             # Algorithm 1 drops "small" displacements; paper doesn't give the value
RES = 256                  # paper / official config
VAL_FRACTION = 0.05

# ---- model ----
INIT = "instruct-pix2pix"  #@param ["instruct-pix2pix", "sd-1.5 (paper)"]
USE_WARP = True  #@param {type:"boolean"}
# USE_WARP=False is the paper's "Pix2Gif-Motion-embed" ablation (no FlowNet/WarpNet, 8-ch UNet input)
PERCEPTUAL_WEIGHT = 0.01   # paper λ_P (ddpm_edit.py: per_lambda = 0.01)
UNCOND_P = 0.05            # official CFG dropout: 5% text-only, 5% image-only, 5% both

# ---- training (paper: 16×V100, eff. batch 2×8 accum×16 GPUs = 256, ~24k steps, AdamW 1e-4) ----
MAX_STEPS = 5000  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
GRAD_ACCUM = 2  #@param {type:"integer"}
LR = 1e-4
WARMUP_STEPS = 500         # official: 10k warm-up at eff. batch 256; scaled down
USE_8BIT_ADAM = True  #@param {type:"boolean"}
USE_EMA = True  #@param {type:"boolean"}
SAVE_EVERY = 1000  #@param {type:"integer"}
SEED = 0

MODEL_IDS = {"instruct-pix2pix": "timbrooks/instruct-pix2pix",
             "sd-1.5 (paper)": "stable-diffusion-v1-5/stable-diffusion-v1-5"}
VAE_ID = "stabilityai/sd-vae-ft-mse"  # paper: "ft-MSE autoencoder"
DATA_DIR = "/content/tgif_pairs"
CKPT_PATH = f"{ROOT}/pix2gif_{'warp' if USE_WARP else 'noflow'}_{INIT.split()[0]}.pt"
DEVICE = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("checkpoint ->", CKPT_PATH, "| autocast", DTYPE)

### Dataset
TGIF GIF URLs (tumblr) still resolve as of Oct 2026. Pairs are saved in the official JSON format (`[src, tgt, caption, flow]`, split by GIF), so the official `edit_dataset.py` can read them too.

In [ ]:
#@title 3 · Build the TGIF motion-pair dataset (paper Sec. 3.3 + Algorithm 1)
import io, json, random, tarfile, time
from concurrent.futures import ThreadPoolExecutor
import cv2, numpy as np, requests
from PIL import Image, ImageSequence

TSV_URL = "https://raw.githubusercontent.com/raingo/TGIF-Release/master/data/tgif-v1.0.tsv"
JSON_NAME = "train_val_dataset_motion_2_20_reduced.json"  # same file name/format the official edit_dataset.py reads
TAR_PATH = f"{ROOT}/tgif_pairs_{N_GIFS}.tar"


def flow_magnitude(f0, f1):
    """Algorithm 1: grayscale -> features to track (Shi-Tomasi) -> Lucas-Kanade -> |displacement|,
    drop small values, average."""
    g0, g1 = cv2.cvtColor(f0, cv2.COLOR_RGB2GRAY), cv2.cvtColor(f1, cv2.COLOR_RGB2GRAY)
    p0 = cv2.goodFeaturesToTrack(g0, maxCorners=200, qualityLevel=0.01, minDistance=7, blockSize=7)
    if p0 is None or len(p0) < 5:
        return None
    p1, st, _ = cv2.calcOpticalFlowPyrLK(g0, g1, p0, None, winSize=(15, 15), maxLevel=2,
                                         criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
    if p1 is None:
        return None
    ok = st.reshape(-1) == 1
    if ok.sum() < 5:
        return None
    mag = np.linalg.norm((p1 - p0).reshape(-1, 2)[ok], axis=1)
    mag = mag[mag > FLOW_EPS]
    return float(mag.mean()) if len(mag) else 0.0


def gif_frames(data, max_frames=96):
    im = Image.open(io.BytesIO(data))
    frames = []
    for i, f in enumerate(ImageSequence.Iterator(im)):
        if i >= max_frames:
            break
        # official edit_dataset.py resizes straight to 256×256 (no crop); flow is measured at that scale too
        frames.append(np.asarray(f.convert("RGB").resize((RES, RES), Image.Resampling.LANCZOS)))
    return frames


def process_gif(args):
    gid, url, caption = args
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        frames = gif_frames(r.content)
    except Exception:
        return []
    n = len(frames)
    if n < 2:
        return []
    cands = [(i, j) for i in range(n) for j in range(i + 1, n)]  # forward pairs (source earlier than target)
    random.Random(gid).shuffle(cands)
    kept = []
    for i, j in cands[:80]:
        m = flow_magnitude(frames[i], frames[j])
        if m is not None and MOTION_RANGE[0] <= m <= MOTION_RANGE[1]:
            kept.append((i, j, m))
        if len(kept) >= PAIRS_PER_GIF:
            break
    out = []
    for i, j, m in kept:
        paths = []
        for k in (i, j):
            p = f"{DATA_DIR}/frames/{gid:06d}_{k:03d}.jpg"
            if not os.path.exists(p):
                Image.fromarray(frames[k]).save(p, quality=95)
            paths.append(p)
        out.append([paths[0], paths[1], caption.strip(), round(m, 3), gid])
    return out


def build_dataset():
    os.makedirs(f"{DATA_DIR}/frames", exist_ok=True)
    rows = [l.split("\t") for l in requests.get(TSV_URL, timeout=60).text.strip().split("\n")]
    random.Random(SEED).shuffle(rows)
    jobs = [(i, u, c) for i, (u, c) in enumerate(rows[:N_GIFS])]
    pairs, t0 = [], time.time()
    with ThreadPoolExecutor(32) as ex:
        for k, res in enumerate(ex.map(process_gif, jobs), 1):
            pairs += res
            if k % 250 == 0:
                print(f"  {k}/{len(jobs)} GIFs · {len(pairs)} pairs · {time.time() - t0:.0f}s")
    gids = sorted({p[4] for p in pairs})
    val_gids = set(random.Random(SEED).sample(gids, max(1, int(len(gids) * VAL_FRACTION))))  # split by GIF, no leakage
    split = {"train": [p[:4] for p in pairs if p[4] not in val_gids],
             "val": [p[:4] for p in pairs if p[4] in val_gids]}
    with open(f"{DATA_DIR}/{JSON_NAME}", "w") as f:
        json.dump(split, f)
    with tarfile.open(TAR_PATH, "w") as t:
        t.add(DATA_DIR, arcname=os.path.basename(DATA_DIR))
    return split


if os.path.exists(f"{DATA_DIR}/{JSON_NAME}"):
    split = json.load(open(f"{DATA_DIR}/{JSON_NAME}"))
elif os.path.exists(TAR_PATH):
    print("restoring dataset from", TAR_PATH)
    with tarfile.open(TAR_PATH) as t:
        t.extractall(os.path.dirname(DATA_DIR))
    split = json.load(open(f"{DATA_DIR}/{JSON_NAME}"))
else:
    print(f"downloading {N_GIFS} TGIF GIFs and mining motion pairs (~{N_GIFS // 400 + 1} min)...")
    split = build_dataset()

print(f"train pairs: {len(split['train'])} · val pairs: {len(split['val'])}")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
ax[0].hist([p[3] for p in split["train"]], bins=36)
ax[0].set_title("flow magnitude of training pairs")
for a, (s, t, cap, m) in zip(ax[1:], random.Random(1).sample(split["train"], 3)):
    a.imshow(np.hstack([np.asarray(Image.open(s)), np.asarray(Image.open(t))]))
    a.set_title(f"{cap[:38]}… | flow {m:.1f}", fontsize=8)
    a.axis("off")
plt.show()

In [ ]:
#@title 4 · Model: Pix2Gif modules (ported from the official code) on a diffusers UNet
import inflect
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

_inflect = inflect.engine()


def motion_prompt(caption, motion):
    """Official: caption + ' The optical flow is <words>.'  (words, not digits: CLIP handles them better)."""
    caption = caption.strip()
    caption = caption if caption.endswith(".") else caption + "."
    return f"{caption} The optical flow is {_inflect.number_to_words(int(motion))}."


# ---- util.py ----
def conv(i, o, norm=nn.BatchNorm2d, k=3, stride=1):
    return nn.Sequential(nn.Conv2d(i, o, k, stride, (k - 1) // 2, bias=True), norm(o), nn.LeakyReLU(0.2, inplace=True))

def deconv(i, o, norm):
    return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1, bias=True), norm(o), nn.LeakyReLU(0.2, inplace=True))

def predict_flow(i):
    return nn.Sequential(nn.Conv2d(i, 2, 3, 1, 1, bias=True), nn.Tanh())

def i_conv(i, o, norm, k=3, stride=1, bias=True):
    return nn.Sequential(nn.Conv2d(i, o, k, stride, (k - 1) // 2, bias=bias), norm(o), nn.LeakyReLU(0.2, inplace=True))


# ---- openaimodel.py ----
class IntervalEmbedding(nn.Module):
    """Motion magnitude (int) -> learned 768-d vector, broadcast-added to all 77 CLIP tokens (c_L = c_T + c_M)."""
    def __init__(self, num_embeddings=1024, embedding_dim=768):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)

    def forward(self, interval):
        return self.embedding(interval.long().unsqueeze(-1))  # (B, 1, 768)


class InjectionModule(nn.Module):
    """(B, 77, 768) text+motion condition -> (B, size, 16, 16) spatial feature map for FlowNet."""
    def __init__(self, size):
        super().__init__()
        self.size = size
        self.linear = nn.Conv1d(77, size, kernel_size=3, stride=3, padding=1)
        self.norm = nn.LayerNorm([size ** 2, size])
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.norm(self.linear(x).transpose(1, 2))).transpose(1, 2)
        return x.reshape(x.shape[0], -1, self.size, self.size)


class FlowNet(nn.Module):
    """Predicts a sampling grid (tanh, in [-1, 1]) over the 32×32 source latent, conditioned on c_L."""
    def __init__(self, ngf=64, norm=nn.BatchNorm2d, x=4):
        super().__init__()
        self.conv0 = conv(x, ngf, norm)
        self.conv1 = conv(ngf, ngf, norm, stride=2)
        self.conv1_1 = conv(ngf, ngf * 2, norm)
        self.conv2 = conv(ngf * 2, ngf * 2, norm, stride=2)
        self.conv2_1 = conv(ngf * 2, ngf * 2, norm)
        self.deconv1 = deconv(ngf * 2, ngf // 2, norm)
        self.deconv0 = deconv(ngf // 2 + ngf // 4 + 2, ngf // 4, norm)
        self.inter_conv1 = i_conv(ngf // 2 + ngf // 4 + 2, ngf // 2, norm)
        self.inter_conv0 = i_conv(ngf // 4 + 2, ngf // 4, norm)
        self.predict_flow2 = predict_flow(ngf * 2)
        self.predict_flow1 = predict_flow(ngf // 2)
        self.predict_flow0 = predict_flow(ngf // 4)
        self.upsampled_flow2_to_1 = nn.ConvTranspose2d(2, 2, 4, 2, 1)
        self.upsampled_flow1_to_0 = nn.ConvTranspose2d(2, 2, 4, 2, 1)
        self.injection1 = InjectionModule(ngf // 4)

    def forward(self, x, cond):
        out_conv0 = self.conv0(x)                                   # B×64×32×32
        out_conv1 = self.conv1_1(self.conv1(out_conv0))             # B×128×16×16
        out_conv2 = self.conv2_1(self.conv2(out_conv1))             # B×128×8×8
        flow2 = self.predict_flow2(out_conv2)
        flow2_up = self.upsampled_flow2_to_1(flow2)
        out_deconv1 = self.deconv1(out_conv2)
        concat1 = torch.cat((out_deconv1, flow2_up, self.injection1(cond)), 1)  # condition injected at 16×16
        flow1 = self.predict_flow1(self.inter_conv1(concat1))
        flow1_up = self.upsampled_flow1_to_0(flow1)
        concat0 = torch.cat((self.deconv0(concat1), flow1_up), 1)
        flow0 = self.predict_flow0(self.inter_conv0(concat0))       # B×2×32×32
        return flow0, flow1, flow2


def warpnet(images, flow):  # WarpNet: flow is used directly as the grid_sample grid (as in the official code)
    return F.grid_sample(images, flow.permute(0, 2, 3, 1), mode="bilinear", align_corners=False)


# ---- perceptual.py (VGG19, first conv widened to 4 channels so it runs on latents) ----
class LatentPerceptual(nn.Module):
    def __init__(self, layers=(0, 5, 10, 19, 28)):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features[: max(layers) + 1]
        first = vgg[0]
        new = nn.Conv2d(4, first.out_channels, first.kernel_size, first.stride, first.padding)
        with torch.no_grad():
            new.weight[:, :3] = first.weight
            new.weight[:, 3] = first.weight.mean(dim=1)
            new.bias.copy_(first.bias)
        vgg[0] = new
        for m in vgg:
            if isinstance(m, nn.ReLU):
                m.inplace = False  # features are tapped mid-network; keep them intact for autograd
        self.vgg, self.layers = vgg.eval().requires_grad_(False), set(layers)

    def forward(self, a, b):
        loss = 0.0
        for i, layer in enumerate(self.vgg):
            a, b = layer(a), layer(b)
            if i in self.layers:
                loss = loss + torch.mean((a - b) ** 2)
        return loss


# ---- the model (ddpm_edit.py: get_warped_image + DiffusionWrapper 'hybrid') ----
class Pix2Gif(nn.Module):
    def __init__(self, unet, use_warp=True):
        super().__init__()
        self.unet, self.use_warp = unet, use_warp
        self.interval_embedding = IntervalEmbedding()
        self.flownet = FlowNet(64, x=4)

    def forward(self, noisy, t, text_emb, interval, src_latent):
        ctx = text_emb + self.interval_embedding(interval)        # c_L = c_T + c_M; also the UNet's cross-attn context
        warped = None
        x = [noisy, src_latent]
        if self.use_warp:
            warped = warpnet(src_latent, self.flownet(src_latent, ctx)[0])
            x.append(warped)                                       # "Warp-concat": [z_t, E(c_I), z_W] -> 12 channels
        return self.unet(torch.cat(x, 1), t, encoder_hidden_states=ctx).sample, warped


def widen_conv_in(unet, in_ch):
    """Extra input channels start at zero, so the init model's behaviour is unchanged at step 0 (IP2P trick)."""
    old = unet.conv_in
    new = nn.Conv2d(in_ch, old.out_channels, old.kernel_size, old.stride, old.padding)
    with torch.no_grad():
        new.weight.zero_()
        n = min(in_ch, old.in_channels)
        new.weight[:, :n] = old.weight[:, :n]
        new.bias.copy_(old.bias)
    unet.conv_in = new
    unet.register_to_config(in_channels=in_ch)
    return unet


def load_components():
    from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
    from transformers import CLIPTextModel, CLIPTokenizer
    mid = MODEL_IDS[INIT]
    tok = CLIPTokenizer.from_pretrained(mid, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(mid, subfolder="text_encoder", torch_dtype=DTYPE).to(DEVICE).eval()
    vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=torch.float32).to(DEVICE).eval()
    unet = UNet2DConditionModel.from_pretrained(mid, subfolder="unet")
    sched = DDPMScheduler.from_pretrained(mid, subfolder="scheduler")
    unet = widen_conv_in(unet, 12 if USE_WARP else 8)
    model = Pix2Gif(unet, USE_WARP).to(DEVICE)
    return tok, text_encoder, vae, sched, model


tok, text_encoder, vae, noise_sched, model = load_components()
text_encoder.requires_grad_(False)
vae.requires_grad_(False)
SCALE = vae.config.scaling_factor  # 0.18215


@torch.no_grad()
def encode_text(prompts):
    ids = tok(prompts, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids
    return text_encoder(ids.to(DEVICE))[0].float()


n_train = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Pix2Gif ready · init={INIT} · warp={USE_WARP} · trainable params {n_train:.0f}M "
      f"(UNet + FlowNet {sum(p.numel() for p in model.flownet.parameters()) / 1e6:.2f}M + motion embedding)")

### Training
Re-running this cell resumes from the last checkpoint on Drive. To train longer, raise `MAX_STEPS` in cell 2 and re-run cells 2 and 5. The optimizer state isn't saved (it's about 7 GB), so it restarts on resume.

In [ ]:
#@title 5 · Train (resumes automatically from the Drive checkpoint)
import copy, math
from torch.utils.data import DataLoader, Dataset


class MotionPairs(Dataset):
    """edit_dataset.EditDatasetMotionInterval: (source, target, caption, flow) -> 256² tensors, shared random h-flip."""
    def __init__(self, rows, flip_prob=0.5):
        self.rows, self.flip_prob = rows, flip_prob

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        s, t, cap, m = self.rows[i]
        load = lambda p: torch.from_numpy(np.asarray(
            Image.open(p).convert("RGB").resize((RES, RES), Image.Resampling.LANCZOS))).permute(2, 0, 1).float() / 127.5 - 1
        s, t = load(s), load(t)
        if random.random() < self.flip_prob:
            s, t = s.flip(-1), t.flip(-1)
        return {"src": s, "tgt": t, "prompt": motion_prompt(cap, m), "motion": int(m)}


class EMA:
    """LitEma from the official code: decay = min(0.9999, (1 + n) / (10 + n))."""
    def __init__(self, module):
        self.shadow = {k: v.detach().clone().float() for k, v in module.state_dict().items() if v.dtype.is_floating_point}
        self.n = 0

    @torch.no_grad()
    def update(self, module):
        self.n += 1
        d = min(0.9999, (1 + self.n) / (10 + self.n))
        for k, v in module.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(d).add_(v.float(), alpha=1 - d)


def save_ckpt(step, ema, opt=None):
    state = {"step": step, "model": {k: v.half() if v.dtype == torch.float32 else v for k, v in model.state_dict().items()},
             "ema": {k: v.half() for k, v in ema.shadow.items()} if ema else None,
             "config": {"INIT": INIT, "USE_WARP": USE_WARP, "RES": RES}}
    torch.save(state, CKPT_PATH + ".tmp")
    os.replace(CKPT_PATH + ".tmp", CKPT_PATH)  # atomic: a Colab disconnect can't corrupt it


torch.manual_seed(SEED)
random.seed(SEED)
loader = DataLoader(MotionPairs(split["train"]), batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                    drop_last=True, pin_memory=True, persistent_workers=True)
perceptual = LatentPerceptual().to(DEVICE)
model.unet.enable_gradient_checkpointing()
params = [p for p in model.parameters() if p.requires_grad]
if USE_8BIT_ADAM:
    import bitsandbytes as bnb
    opt = bnb.optim.AdamW8bit(params, lr=LR)
else:
    opt = torch.optim.AdamW(params, lr=LR)
lr_sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / WARMUP_STEPS))
scaler = torch.amp.GradScaler(enabled=DTYPE == torch.float16)
ema = EMA(model) if USE_EMA else None
null_emb = encode_text([""])
step = 0
if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location="cpu")
    model.load_state_dict({k: v.float() if v.dtype == torch.float16 else v for k, v in ck["model"].items()})
    if ema and ck.get("ema"):
        ema.shadow = {k: v.float().to(DEVICE) for k, v in ck["ema"].items()}
        ema.n = ck["step"]
    step = ck["step"]
    for _ in range(step):
        lr_sched.step()
    print(f"resumed from step {step} (optimizer state restarts)")

model.train()
log, t0, start_step = [], time.time(), step
while step < MAX_STEPS:
    for i, b in enumerate(loader):
        src, tgt = b["src"].to(DEVICE, non_blocking=True), b["tgt"].to(DEVICE, non_blocking=True)
        bs = src.shape[0]
        with torch.no_grad():
            z = vae.encode(tgt).latent_dist.sample() * SCALE          # target: scaled sample (get_first_stage_encoding)
            src_lat = vae.encode(src).latent_dist.mode()             # condition: unscaled mode (encode_first_stage().mode())
            text = encode_text(b["prompt"])
            r = torch.rand(bs, device=DEVICE)                        # official CFG dropout
            text = torch.where((r < 2 * UNCOND_P)[:, None, None], null_emb, text)
            src_lat = src_lat * (1 - ((r >= UNCOND_P) & (r < 3 * UNCOND_P)).float())[:, None, None, None]
            noise = torch.randn_like(z)
            t = torch.randint(0, noise_sched.config.num_train_timesteps, (bs,), device=DEVICE)
            noisy = noise_sched.add_noise(z, noise, t)
        with torch.autocast(DEVICE, dtype=DTYPE):
            pred, warped = model(noisy, t, text, b["motion"].to(DEVICE), src_lat)
        loss_ldm = F.mse_loss(pred.float(), noise)
        loss_p = perceptual(warped.float(), src_lat) if USE_WARP else torch.zeros((), device=DEVICE)
        loss = (loss_ldm + PERCEPTUAL_WEIGHT * loss_p) / GRAD_ACCUM   # L_T = L'_LDM + λ_P · L_P
        scaler.scale(loss).backward()
        if (i + 1) % GRAD_ACCUM:
            continue
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        lr_sched.step()
        ema and ema.update(model)
        step += 1
        log.append((loss_ldm.item(), loss_p.item()))
        if step % 50 == 0:
            l = np.mean(log[-50:], axis=0)
            print(f"step {step:>6}/{MAX_STEPS} · ldm {l[0]:.4f} · perceptual {l[1]:.4f} · "
                  f"{(time.time() - t0) / (step - start_step):.2f}s/step")
        if step % SAVE_EVERY == 0 or step == MAX_STEPS:
            save_ckpt(step, ema)
            print("  saved", CKPT_PATH)
        if step >= MAX_STEPS:
            break
model.eval()

### Generate
Pix2Gif is **image → GIF**. For text-only input, pick `text (sd-turbo)` to create the first frame. Larger or more spread-out motion values give bigger movement. `IMAGE_CFG` in 1.6–2.2 is the paper's best range: higher sticks closer to the source.

In [ ]:
#@title 6 · Generate a GIF (demo.py sampling: Euler ancestral, 3-way CFG, one frame per motion value)
from diffusers import EulerAncestralDiscreteScheduler
from IPython.display import HTML, display
import base64

USE_EMA_WEIGHTS = True  #@param {type:"boolean"}


def load_for_inference(path=CKPT_PATH, use_ema=USE_EMA_WEIGHTS):
    if not os.path.exists(path):
        print("no checkpoint yet: using the current in-memory weights")
        return
    ck = torch.load(path, map_location="cpu")
    sd = ck["ema"] if use_ema and ck.get("ema") else ck["model"]
    full = model.state_dict()
    full.update({k: v.to(full[k].dtype) for k, v in sd.items() if k in full})
    model.load_state_dict(full)
    print(f"loaded step {ck['step']} ({'EMA' if use_ema and ck.get('ema') else 'raw'} weights)")


@torch.no_grad()
def pix2gif(image, caption, motions=(2, 4, 6, 8, 11, 13, 16, 19), steps=100, seed=1371,
            text_cfg=7.5, image_cfg=1.8, shared_noise=True):
    """Every frame is generated from the SAME source image, with a different motion value (as in demo.py).
    shared_noise=True re-seeds per frame like demo.py (more consistent frames); False = demo_parallel.py."""
    model.eval()
    w, h = image.size
    src = torch.from_numpy(np.asarray(image.convert("RGB").resize((RES, RES), Image.Resampling.LANCZOS)))
    src = (src.permute(2, 0, 1).float() / 127.5 - 1)[None].to(DEVICE)
    src_lat = vae.encode(src).latent_dist.mode()
    null = encode_text([""])
    sch = EulerAncestralDiscreteScheduler.from_config(noise_sched.config, timestep_spacing="linspace")
    frames = []
    for k, m in enumerate(motions):
        g = torch.Generator(DEVICE).manual_seed(seed if shared_noise else seed + k)
        cond = encode_text([motion_prompt(caption, m)])
        txt = torch.cat([cond, null, null])                                    # [cond, img-only, uncond]
        img = torch.cat([src_lat, src_lat, torch.zeros_like(src_lat)])
        iv = torch.full((3,), int(m), device=DEVICE)                           # motion embedding on all 3 branches
        sch.set_timesteps(steps, device=DEVICE)
        lat = torch.randn(src_lat.shape, generator=g, device=DEVICE) * sch.init_noise_sigma
        for t in sch.timesteps:
            inp = sch.scale_model_input(lat, t).repeat(3, 1, 1, 1)
            with torch.autocast(DEVICE, dtype=DTYPE):
                eps, _ = model(inp, t.expand(3), txt, iv, img)
            e_c, e_i, e_u = eps.float().chunk(3)
            eps = e_u + text_cfg * (e_c - e_i) + image_cfg * (e_i - e_u)
            lat = sch.step(eps, t, lat, generator=g).prev_sample
        x = vae.decode(lat / SCALE).sample.clamp(-1, 1)
        x = ((x[0].permute(1, 2, 0) + 1) * 127.5).round().byte().cpu().numpy()
        frames.append(Image.fromarray(x).resize((w, h), Image.Resampling.LANCZOS))
    return frames


def save_gif(frames, path, fps=4, boomerang=False):
    seq = frames + frames[-2:0:-1] if boomerang else frames
    seq[0].save(path, save_all=True, append_images=seq[1:], duration=int(1000 / fps), loop=0)
    return path


def show(path, width=320):
    b = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f"<img src='data:image/gif;base64,{b}' width={width}>"))


load_for_inference()

SOURCE = "val"  #@param ["val", "upload", "url", "text (sd-turbo)"]
URL = ""  #@param {type:"string"}
CAPTION = "a cat is dancing."  #@param {type:"string"}
MOTIONS = "2,4,6,8,11,13,16,19"  #@param {type:"string"}
STEPS = 50  #@param {type:"slider", min:20, max:100, step:5}
TEXT_CFG = 7.5  #@param {type:"number"}
IMAGE_CFG = 1.8  #@param {type:"number"}
FPS = 4  #@param {type:"integer"}
BOOMERANG = False  #@param {type:"boolean"}

if SOURCE == "val":
    s, _, cap, _ = random.choice(split["val"])
    image, CAPTION = Image.open(s).convert("RGB"), cap
elif SOURCE == "upload":
    from google.colab import files
    image = Image.open(io.BytesIO(next(iter(files.upload().values())))).convert("RGB")
elif SOURCE == "url":
    image = Image.open(io.BytesIO(requests.get(URL, timeout=30).content)).convert("RGB")
else:  # Pix2Gif is image-to-GIF; for text-only input, make the first frame with a fast T2I model
    from diffusers import AutoPipelineForText2Image
    t2i = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16).to(DEVICE)
    image = t2i(CAPTION, num_inference_steps=1, guidance_scale=0.0).images[0]
    del t2i
    torch.cuda.empty_cache()

frames = pix2gif(image, CAPTION, [int(x) for x in MOTIONS.split(",")], steps=STEPS,
                 text_cfg=TEXT_CFG, image_cfg=IMAGE_CFG)
out = save_gif(frames, f"{ROOT}/sample_{int(time.time())}.gif", fps=FPS, boomerang=BOOMERANG)
print(CAPTION, "->", out)
image.resize((256, int(256 * image.height / image.width))).save("/content/_src.gif")
show("/content/_src.gif", 256)
show(out, 256)

### Evaluate
This covers the paper's **motion coherency** (L2 and Pearson correlation between the requested motion and the LK flow measured between the source and each generated frame) and **CLIPSim**, on held-out TGIF GIFs. FVD and PIC need UCF-101 / MSR-VTT. The official scripts (`calculate_fvd.py`, `metrics/`) are in `/content/Pix2Gif-official`.

In [ ]:
#@title 7 · Evaluate (paper Sec. 4: motion coherency L2 / PCC, CLIPSim)
from transformers import CLIPModel, CLIPProcessor

N_EVAL = 30  #@param {type:"integer"}
EVAL_MOTIONS = [2, 4, 6, 8, 11, 14, 17, 19]  # the paper's ablation motion list

clip = CLIPModel.from_pretrained("openai/clip-vit-large-patch14", torch_dtype=torch.float16).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")


@torch.no_grad()
def clip_sim(frames, caption):
    inp = clip_proc(text=[caption], images=frames, return_tensors="pt", padding=True).to(DEVICE)
    out = clip(**{k: v.half() if v.dtype == torch.float32 else v for k, v in inp.items()})
    i = F.normalize(out.image_embeds.float(), dim=-1)
    t = F.normalize(out.text_embeds.float(), dim=-1)
    return float((i @ t.T).mean())


rows_eval = random.Random(0).sample(split["val"], min(N_EVAL, len(split["val"])))
asked, measured, sims = [], [], []
for k, (s, _, cap, _) in enumerate(rows_eval):
    img = Image.open(s).convert("RGB")
    frames = pix2gif(img, cap, EVAL_MOTIONS, steps=STEPS, text_cfg=TEXT_CFG, image_cfg=IMAGE_CFG)
    src = np.asarray(img.resize((RES, RES)))
    for m, f in zip(EVAL_MOTIONS, frames):
        mm = flow_magnitude(src, np.asarray(f.resize((RES, RES))))  # same Algorithm 1 used to label the data
        if mm is not None:
            asked.append(m)
            measured.append(mm)
    sims.append(clip_sim(frames, cap))
    if k < 4:
        show(save_gif(frames, f"/content/eval_{k}.gif"), 192)

asked, measured = np.array(asked, float), np.array(measured, float)
print(f"motion coherency  L2 = {np.sqrt(np.mean((asked - measured) ** 2)):.2f}  "
      f"PCC = {np.corrcoef(asked, measured)[0, 1]:.3f}   (n={len(asked)} frames)")
print(f"CLIPSim (frames vs caption) = {np.mean(sims):.4f}")
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(asked, measured, s=8, alpha=0.5)
ax.plot([0, 20], [0, 20], "k--", lw=1)
ax.set_xlabel("input motion")
ax.set_ylabel("measured flow (LK)")
ax.set_title("motion control")
plt.show()